# Day I4 — 운동역학: 지면반력, 관절 모멘트, 파워

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/I-gait/day-I4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/I-gait/day-I4.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 한 발로 정지해 선 사람의 발목 모멘트 (가상 숫자)

In [ ]:
import numpy as np

# 정지 상태에서 한 발로 선 사람의 발 분절 자유물체도 (가상 숫자)
# 좌표: x = 앞(+), y = 위(+), 반시계 모멘트 = +.  원점 = 발목 관절 중심
g = 9.81
m_body, m_foot = 60.0, 0.9            # kg
grf = np.array([0.0, m_body * g])     # 지면이 발을 미는 힘(N). 정지 → 체중과 같다
r_cop = np.array([0.04, -0.07])       # COP: 발목보다 4 cm 앞, 7 cm 아래(바닥)
r_com = np.array([0.025, -0.04])      # 발 질량중심

def cross2(r, f):                     # 2D 외적의 z 성분 = r_x F_y − r_y F_x
    return r[0] * f[1] - r[1] * f[0]

# 정지: ΣF = 0, 발목 기준 ΣM = 0  →  발목이 발에 주는 힘 F_a와 모멘트 M_a
F_a = -grf - np.array([0.0, -m_foot * g])
M_a = -(cross2(r_cop, grf) + cross2(r_com, np.array([0.0, -m_foot * g])))
print("발목 반력(다리가 발에 주는 힘) = (%.1f, %.1f) N" % tuple(F_a))
print("발목 내부 모멘트 = %.1f N·m  →  %s" % (M_a, "시계 방향 = 족저굴곡근(plantarflexor) 모멘트" if M_a < 0 else "반시계 = 배측굴곡근 모멘트"))
print("체중으로 나누면 %.2f N·m/kg" % (abs(M_a) / m_body))
print("외부 모멘트(지면반력이 발목에 만드는 것) = %.1f N·m → 발목을 배측굴곡시키려는 '요구'" % cross2(r_cop, grf))

### 예시 3 — 실제 데이터: gaitpdb 깔창 센서의 수직 지면반력

> **데이터:** Hausdorff JM. *Gait in Parkinson's Disease* (v1.0.0). PhysioNet; 2008. [doi:10.13026/C24H3N](https://doi.org/10.13026/C24H3N). 라이선스 Open Data Commons Attribution License v1.0. PhysioNet 표준 인용: Pollard T, et al. PhysioNet as a global platform for biomedical research. *Nature Health* (2026). [doi:10.1038/s44360-026-00096-z](https://doi.org/10.1038/s44360-026-00096-z). 신호 파일은 19열(1열 시간, 18열 왼발 8개 센서 합, 19열 오른발 합, 단위 N, 100 Hz)입니다.
>
> ```bash
> mkdir -p data/gaitpdb && cd data/gaitpdb
> curl -O https://physionet.org/files/gaitpdb/1.0.0/GaCo15_01.txt
> curl -O https://physionet.org/files/gaitpdb/1.0.0/GaCo01_01.txt
> curl -O https://physionet.org/files/gaitpdb/1.0.0/demographics.txt
> cd ../..
> ```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

fs = 100                                                     # gaitpdb 표본화(Hz)
demo = pd.read_csv("data/gaitpdb/demographics.txt", sep="\t", usecols=range(20)).set_index("ID")

def stances(force, thr=20):
    """force > thr 인 연속 구간의 시작·끝 표본 번호(완전한 구간만)"""
    on = (force > thr).astype(int)
    st = np.flatnonzero(np.diff(on) == 1) + 1
    en = np.flatnonzero(np.diff(on) == -1) + 1
    st = st[st < en[-1]]
    en = en[np.searchsorted(en, st[0]):][: len(st)]
    return st, en

def stance_curves(force, bw, trim=5):
    st, en = stances(force)
    stance_pct = np.median((en - st)[:-1] / np.diff(st))     # stance가 주기의 몇 %인가
    grid = np.linspace(0, 100, 101)
    curves = np.array([np.interp(grid, np.linspace(0, 100, e - s), force[s:e] / bw)
                       for s, e in zip(st[trim:-trim], en[trim:-trim])])   # 앞뒤 5개는 버림(가속·감속)
    return grid * stance_pct, curves                          # 가로축을 %GC로

sid = "GaCo15"                                                # 대조군 한 명, walk 01(평소 걷기)
x = np.loadtxt(f"data/gaitpdb/{sid}_01.txt")
bw = demo.loc[sid, "Weight"] * 9.81
print("%s: 평소 속도 %.2f m/s, 체중 %.0f N, 기록 %.0f s" % (sid, demo.loc[sid, "Speed_01"], bw, len(x) / fs))
print("점검: 두 발 합의 평균 / 체중 = %.2f  (힘판이라면 1.00에 가까워야 한다)" % (x[:, 17:19].sum(axis=1).mean() / bw))

fig, ax = plt.subplots(figsize=(6.5, 3.6))
for col, side, color in [(17, "left", "tab:blue"), (18, "right", "tab:orange")]:
    gc, C = stance_curves(x[:, col], bw)
    m = C.mean(axis=0)
    pk, _ = find_peaks(m, prominence=0.05)
    f1, f3 = pk[0], pk[-1]
    f2 = f1 + np.argmin(m[f1:f3 + 1])
    print("%-5s n=%d  F1 %.2f BW @%2.0f%%GC | F2 %.2f BW @%2.0f%%GC | F3 %.2f BW @%2.0f%%GC | stance 끝 %2.0f%%GC"
          % (side, len(C), m[f1], gc[f1], m[f2], gc[f2], m[f3], gc[f3], gc[-1]))
    ax.plot(gc, m, color=color, lw=2, label=side)
    for name, i in [("F1", f1), ("F2", f2), ("F3", f3)]:
        ax.annotate(name, (gc[i], m[i]), textcoords="offset points", xytext=(0, 6), ha="center", color=color)
ax.axhline(1, color="k", ls=":", lw=1)
ax.set_xlabel("% gait cycle (initial contact = 0)")
ax.set_ylabel("insole vertical force / body weight")
ax.set_title("gaitpdb v1.0.0, one control, usual walk (mean of stances)")
ax.legend()
out = Path("figures/advanced/I/I4_vgrf.png")
out.parent.mkdir(parents=True, exist_ok=True)
fig.tight_layout(); fig.savefig(out, dpi=150)
print("저장:", out)